# 인사 데이터 JSONL 변환 (OpenSearch 적재용)

### 라이브러리 선언

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
import os
from dotenv import load_dotenv

print(f'pandas 버전: {pd.__version__}')
print('라이브러리 로딩 완료!')

### 환경 설정 *** 경로 확인 필요

In [ ]:
load_dotenv()

INPUT_DIR  = Path(os.getenv('INPUT_DIR',  '../데이터 전처리 정제/output'))
OUTPUT_DIR = Path(os.getenv('OUTPUT_DIR', 'output'))

LICENSE = os.getenv('DATA_LICENSE', 'Internal Use Only')
VERSION = os.getenv('DATA_VERSION', '1.0')

os.makedirs(OUTPUT_DIR, exist_ok=True)

print('경로 설정:')
print(f'  입력 디렉토리: {INPUT_DIR}')
print(f'  출력 디렉토리: {OUTPUT_DIR}')

### 인덱스별 필드 매핑 정의

In [ ]:
# 인덱스별 포함 필드 (데이터구조정의서 v1.4 기준)
INDEX_FIELD_MAP = {
    'hr_basic_1': [
        '사원번호','이름','성별','나이','입사일','근속기간',
        '채용경로','계약형태','회사명','사업장위치',
        '부서','팀','부서레벨','직급','직책','직급레벨','이메일'
    ],
    'hr_basic_2': [
        '사원번호','생년월일','병역','학력','출신대학','학점',
        '전화번호','이전직장명','이전최종직급','이전담당업무'
    ],
    'hr_basic_3': [
        '사원번호','주민등록번호','주소','퇴직구분','퇴직일자'
    ],
    'hr_performance_2': [
        '사원번호','성과점수',
        '인사고과_2020','인사고과_2021','인사고과_2022','인사고과_2023','인사고과_2024',
        '자격증','TOEIC점수','포상이력'
    ],
    'hr_performance_3': [
        '사원번호','징계이력','징계사유','자격증수당여부'
    ],
    'hr_salary_2': [
        '사원번호','잔업시간','미사용휴가일수'
    ],
    'hr_salary_3': [
        '사원번호','연봉','급여은행','계좌번호','4대보험가입여부'
    ],
}

# 원본 CSV → 생성할 인덱스 목록
SOURCE_INDEX_MAP = {
    '기본인사정보_정제': ['hr_basic_1', 'hr_basic_2', 'hr_basic_3'],
    '역량성과_정제':     ['hr_performance_2', 'hr_performance_3'],
    '급여정보_정제':     ['hr_salary_2', 'hr_salary_3'],
}

# _id 유형코드
ID_PREFIX_MAP = {
    'hr_basic_1': 'BAS',  'hr_basic_2': 'BAS',  'hr_basic_3': 'BAS',
    'hr_performance_2': 'PERF', 'hr_performance_3': 'PERF',
    'hr_salary_2': 'SAL', 'hr_salary_3': 'SAL',
}

print('인덱스 매핑 정의 완료!')
print(f'  총 인덱스 수: {len(INDEX_FIELD_MAP)}개')

# 1. 데이터 로딩

In [ ]:
dfs = {}
for path in sorted(INPUT_DIR.glob('*.csv')):
    df = pd.read_csv(path, encoding='utf-8-sig')
    dfs[path.stem] = df
    print(f'  로딩: {path.name}  ({len(df):,}행 / {len(df.columns)}열)')

# 역량성과, 급여정보의 embedding_text 생성을 위해 기본인사정보 조회용 딕셔너리 구성
basic_df = dfs.get('기본인사정보_정제', pd.DataFrame())
basic_lookup = {}
if not basic_df.empty:
    for _, row in basic_df.iterrows():
        emp_id = str(row.get('사원번호', '')).strip()
        basic_lookup[emp_id] = {
            '이름': str(row.get('이름', '')),
            '부서': str(row.get('부서', '')),
            '직급': str(row.get('직급', '')),
        }

print(f'\n로딩 완료! 총 {len(dfs)}개 파일')
print(f'기본인사정보 조회 딕셔너리: {len(basic_lookup):,}건')

# 2. 변환 함수 정의

In [ ]:
MISSING_VALUES = {'', '미입력', 'nan', 'NaN', 'None', 'none'}

def clean(val) -> str:
    s = str(val).strip()
    return s if s not in MISSING_VALUES else ''


def build_embedding_text(row: pd.Series, index_name: str, info: dict) -> str:
    name = info.get('이름', '')
    dept = info.get('부서', clean(row.get('부서', '')))
    pos  = info.get('직급', clean(row.get('직급', '')))

    field_groups = {
        'hr_basic_1':       [name, clean(row.get('부서','')), clean(row.get('팀','')),
                             clean(row.get('직급','')), clean(row.get('직책','')),
                             clean(row.get('입사일','')), clean(row.get('이메일',''))],
        'hr_basic_2':       [name, clean(row.get('학력','')), clean(row.get('출신대학','')),
                             clean(row.get('전화번호',''))],
        'hr_basic_3':       [name, clean(row.get('주소','')), clean(row.get('퇴직구분',''))],
        'hr_performance_2': [name, dept, pos, clean(row.get('성과점수','')),
                             clean(row.get('자격증','')), clean(row.get('포상이력',''))],
        'hr_performance_3': [name, dept, clean(row.get('징계이력','')),
                             clean(row.get('징계사유',''))],
        'hr_salary_2':      [name, dept, pos, clean(row.get('잔업시간','')),
                             clean(row.get('미사용휴가일수',''))],
        'hr_salary_3':      [name, dept, pos, clean(row.get('연봉','')),
                             clean(row.get('급여은행',''))],
    }
    parts = field_groups.get(index_name, [name, dept, pos])
    return ' '.join(p for p in parts if p)


def to_record(row: pd.Series, index_name: str, seq: int, info: dict) -> dict:
    fields = INDEX_FIELD_MAP[index_name]
    prefix = ID_PREFIX_MAP[index_name]
    emp_id = str(row.get('사원번호', '')).strip()

    record = {
        '_id':              f'{prefix}_{seq:05d}',
        'employee_id':      emp_id,
        'department':       info.get('부서', clean(row.get('부서', ''))),
        'position':         info.get('직급', clean(row.get('직급', ''))),
        'embedding_text':   build_embedding_text(row, index_name, info),
        'embedding_vector': [],
    }

    for f in fields:
        if f in row.index:
            record[f] = row[f] if pd.notna(row.get(f)) else None

    return record


print('변환 함수 정의 완료!')

# 3. JSONL 변환 및 저장

In [ ]:
saved_files = []

for source_name, index_names in SOURCE_INDEX_MAP.items():
    df = dfs.get(source_name)
    if df is None:
        print(f'[SKIP] {source_name}.csv 없음')
        continue

    for index_name in index_names:
        out_path = OUTPUT_DIR / f'{index_name}.jsonl'
        count = 0
        with open(out_path, 'w', encoding='utf-8') as f:
            for seq, (_, row) in enumerate(df.iterrows(), start=1):
                emp_id = str(row.get('사원번호', '')).strip()
                info   = basic_lookup.get(emp_id, {})
                record = to_record(row, index_name, seq, info)
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                count += 1
        saved_files.append((out_path, count))
        print(f'  저장: {out_path.name:<30} ({count:,}건)')

# 4. 저장 결과 확인

In [ ]:
for out_path, _ in saved_files:
    with open(out_path, 'r', encoding='utf-8') as f:
        sample = json.loads(f.readline())
    print(f'\n[{out_path.name}] 첫 번째 레코드:')
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()

# 5. 요약

In [ ]:
print('=' * 60)
print('JSONL 변환 완료')
print('=' * 60)
for path, count in saved_files:
    print(f'  {path.name:<30} {count:>6,}건')
print('-' * 60)
print(f'총 {len(saved_files)}개 인덱스 파일 생성')